# SMS Spam Classification using SVM (Balanced Dataset & 5-Fold Cross Validation)
Dataset: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset?resource=download

In [1]:
# Install (if needed)
!pip -q install wordcloud nltk scikit-learn


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
import re, string, nltk
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score,classification_report,
confusion_matrix,ConfusionMatrixDisplay,
precision_score,recall_score,f1_score,roc_curve,auc)
from sklearn.utils import resample

nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Upload dataset (`spam.csv`) downloaded from Kaggle.

In [3]:
# Load dataset file (local or via Google Colab upload)
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    pass


Saving spam.csv to spam.csv


In [4]:
df = pd.read_csv("spam.csv", encoding="latin-1")

df = df[['v1', 'v2']]

df.columns = ['label', 'message']


In [5]:
print(df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
print(df['label'].value_counts())


(5572, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None
label      0
message    0
dtype: int64
Duplicates: 403
label
ham     4825
spam     747
Name: count, dtype: int64


## Exploratory Data Analysis (Original Imbalanced Dataset)

In [6]:
df['char_count']=df['message'].str.len()
df['word_count']=df['message'].str.split().str.len()

fig,ax=plt.subplots(1,2,figsize=(10,4))
sns.countplot(data=df,x='label',ax=ax[0])
sns.histplot(data=df,x='word_count',hue='label',bins=30,ax=ax[1])
plt.tight_layout()
plt.show()


<Figure size 1000x400 with 2 Axes>

In [7]:
spam=' '.join(df[df.label=='spam'].message)
ham=' '.join(df[df.label=='ham'].message)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color='white').generate(spam))
plt.axis('off'); plt.title("Spam")
plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color='white').generate(ham))
plt.axis('off'); plt.title("Ham")
plt.show()


<Figure size 1200x500 with 2 Axes>

## Preprocessing

In [8]:
# Load the list of English stopwords (e.g., "the", "is", "and")
stop_words = set(stopwords.words('english'))

# Create a WordNet lemmatizer object to convert words to their base form
lemm = WordNetLemmatizer()

# Define a function to clean and preprocess SMS text
def clean_text(text):

    # Convert all characters to lowercase
    text = str(text).lower()

    # Remove URLs (e.g., http://example.com or www.example.com)
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove email addresses (e.g., abc@gmail.com)
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove all numeric digits from the text
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation marks such as !, ?, ., ,, :, etc.
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Split the sentence into words,
    # remove stopwords,
    # and convert each word to its base (lemma) form
    tokens = [
        lemm.lemmatize(word)
        for word in text.split()
        if word not in stop_words
    ]

    # Join the cleaned words back into a single string
    return ' '.join(tokens)

# Remove duplicate SMS messages from the dataset
df = df.drop_duplicates()

# Apply the clean_text() function to every message
# and store the cleaned text in a new column named 'clean_message'
df['clean_message'] = df['message'].apply(clean_text)

# Convert categorical labels into numerical values
# ham  -> 0
# spam -> 1
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

# Display the first five rows of the processed dataset
df.head()


   label                                                                                                                                                      message                                                                                          clean_message
0      0                                              Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...                     go jurong point crazy available bugis n great world la e buffet cine got amore wat
1      0                                                                                                                                Ok lar... Joking wif u oni...                                                                                ok lar joking wif u oni
2      1  Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's  free entry wkly comp win f

## Balancing Dataset (Oversampling Spam Data)
In the original dataset after deduplication, there are **4,516 Ham messages** and **653 Spam messages** (~87.4% Ham vs 12.6% Spam).
To fix this class imbalance, we oversample the minority Spam class using `resample()` so that both classes have exactly **4,516 samples each** (Total: 9,032 messages).

In [9]:
# Separate majority (Ham) and minority (Spam) classes
df_ham = df[df.label == 0]
df_spam = df[df.label == 1]

# Oversample spam instances to match the 4,516 ham instances
df_spam_oversampled = resample(
    df_spam,
    replace=True,
    n_samples=len(df_ham),
    random_state=42
)

# Combine ham and oversampled spam data into a balanced dataframe
df_balanced = pd.concat([df_ham, df_spam_oversampled]).reset_index(drop=True)

print("--- Original Class Counts ---")
print(f"Ham (0) : {len(df_ham)}")
print(f"Spam (1): {len(df_spam)}")
print(f"Total   : {len(df)}")

print("\n--- Balanced Class Counts (After Oversampling Spam Data) ---")
print(f"Ham (0) : {sum(df_balanced.label == 0)}")
print(f"Spam (1): {sum(df_balanced.label == 1)}")
print(f"Total   : {len(df_balanced)}")


--- Original Class Counts ---
Ham (0) : 4516
Spam (1): 653
Total   : 5169

--- Balanced Class Counts (After Oversampling Spam Data) ---
Ham (0) : 4516
Spam (1): 4516
Total   : 9032


### Visualizing Balanced Class Distribution
Below plot shows the class distribution after adding spam data through oversampling (4,516 Ham vs 4,516 Spam):

In [10]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_balanced, x='label')
plt.xticks([0, 1], ['ham (4516)', 'spam (4516)'])
plt.title("Balanced Dataset Distribution (After Oversampling Spam)", fontweight='bold')
plt.ylabel("Count")
plt.show()


<Figure size 600x400 with 1 Axes>

## Train/Test Split
We split the 9,032 balanced messages into **80% Training (7,225 samples)** and **20% Testing (1,807 samples)**.

In [11]:
X = df_balanced['clean_message']
y = df_balanced['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Set Shape: {X_train.shape} (Ham: {sum(y_train==0)}, Spam: {sum(y_train==1)})")
print(f"Testing Set Shape : {X_test.shape} (Ham: {sum(y_test==0)}, Spam: {sum(y_test==1)})")


Training Set Shape: (7225,) (Ham: 3612, Spam: 3613)
Testing Set Shape : (1807,) (Ham: 904, Spam: 903)


## SVM Model

In [12]:
pipeline=Pipeline([
('tfidf',TfidfVectorizer(max_features=5000,ngram_range=(1,2))),
('svm',LinearSVC())
])

pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)

print("Accuracy:",accuracy_score(y_test,pred))
print("Precision:",precision_score(y_test,pred))
print("Recall:",recall_score(y_test,pred))
print("F1:",f1_score(y_test,pred))
print(classification_report(y_test,pred))


Accuracy: 0.993912562257886
Precision: 0.9879649890590809
Recall: 1.0
F1: 0.9939460649422125
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       904
           1       0.99      1.00      0.99       903

    accuracy                           0.99      1807
   macro avg       0.99      0.99      0.99      1807
weighted avg       0.99      0.99      0.99      1807



## Hyperparameter Tuning

In [13]:
param_grid={
'svm__C':[0.1,1,10]
}
grid=GridSearchCV(pipeline,param_grid,cv=5)
grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)
print("Best CV Score:",grid.best_score_)


Best Parameters: {'svm__C': 10}
Best CV Score: 0.9946020761245673


## 5-Fold Cross Validation

In [14]:
scores=cross_val_score(grid.best_estimator_,df_balanced['clean_message'],df_balanced['label'],cv=5)
print(scores)
print("Mean Accuracy:",scores.mean())


[0.99667958 0.99778639 0.99557032 0.99557032 0.99058693]
Mean Accuracy: 0.9952387080879637


## Prediction Function

In [15]:
model=grid.best_estimator_

def predict_sms(text):
    pred=model.predict([clean_text(text)])[0]
    return "Spam" if pred==1 else "Ham"

print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))


Spam
Ham
